# CPU vs GPU Inference Benchmark on Spectrograms

This notebook loads the trained multiclass spectrogram model and benchmarks inference on a small subset of `.mat` spectrograms.

We compare:
- CPU-only inference
- GPU inference (with optional AMP)

Outputs: total time, throughput (files/s), per-batch latency, and a short summary table.


In [1]:
# Auto-reload and imports
%load_ext autoreload
%autoreload 2

import sys, os, time, pickle
from pathlib import Path
from statistics import mean

import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

# visualization optional (not central to benchmarking)
import matplotlib.pyplot as plt

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Add repo root to path
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.ssamba.utilities.training_utils import create_model
from src.ssamba.utilities.inference_utils import (
    build_mat_dataloader,
    build_mat_dataloader_h5like,
    run_inference_multiclass,
)
from src.ssamba.utilities.spectrogram_utils import (
    load_mat_spectrogram,
    normalize_spectrogram,
    resize_to_target,
    preprocess_to_tensor,
    preprocess_to_tensor_ctf,
)



Python: 3.10.12 (main, Jan 17 2025, 14:35:34) [GCC 11.4.0]
CUDA available: True


/home/sbialek/ONC/selfsupervision_anomalies_onc/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/sbialek/ONC/selfsupervision_anomalies_onc/.venv/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


VisionMamba imported successfully
RMSNorm imported successfully


In [2]:
# Config

# Point to the same dataset and model as the reference notebook
MAT_FILES_DIR = "/mnt/z/CIOOS_Anomaly_detection_work/Burrard_Inlet/testset/ICLISTENHF1354/sampling_2023-11-18_to_2024-01-16/mat/processed"
MODEL_DIR = Path("/home/sbialek/ONC/selfsupervision_anomalies_onc/data/trained-model/finetune/amba-base-f16-t16-b16-lr1e-4-m300-custom-tr0.8-fir_experiment_multiclass_avgtok2")
CHECKPOINT_PATH = MODEL_DIR / "models/ft-avgtok_best_checkpoint.pth"
OUTPUT_DIR = Path("benchmark_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# spectrogram/image params
TARGET_SIZE = (512, 512)
EXPECTED_SHAPE = (854, 1000)

# small subset for timing
SUBSET_SIZE = 128  # adjust if you want an even smaller run
BATCH_SIZE = 16
NUM_WORKERS = 4
PIN_MEMORY = True

print("Config ready.")


Config ready.


In [3]:
# Collect a small subset of .mat files and build loaders
from pathlib import Path

all_paths = sorted([str(p) for p in Path(MAT_FILES_DIR).glob('**/*.mat')])
if len(all_paths) == 0:
    raise RuntimeError(f"No .mat files found under {MAT_FILES_DIR}")

subset_paths = all_paths[:min(SUBSET_SIZE, len(all_paths))]
print(f"Subset size: {len(subset_paths)}")

# Two preprocessing pipelines exist; we'll use the 'h5_like' one for consistency
cpu_loader = build_mat_dataloader_h5like(
    paths=subset_paths,
    expected_shape=EXPECTED_SHAPE,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=0,            # workers=0 on CPU to keep it simple
)

gpu_loader = build_mat_dataloader_h5like(
    paths=subset_paths,
    expected_shape=EXPECTED_SHAPE,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    dataset_mean=None,
    dataset_std=None,
    amount=1.0,
)


Subset size: 128


In [ ]:
# Load model once; we will run it on CPU and GPU separately
with open(MODEL_DIR / 'args.pkl', 'rb') as f:
    margs = pickle.load(f)

margs.multiclass = True
margs.exp_dir = MODEL_DIR

# build model on CPU first
cpu_device = torch.device('cpu')
model_cpu = create_model(margs).to(cpu_device)

# load checkpoint weights
ckpt = torch.load(CHECKPOINT_PATH, map_location=cpu_device)
state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
model_cpu.load_state_dict(state_dict)
model_cpu.eval()

# Disable Mamba fast path on CPU (fast path uses CUDA-only kernels)
try:
    from mamba_ssm.modules.mamba_simple import Mamba
    for mod in model_cpu.modules():
        if isinstance(mod, Mamba):
            mod.use_fast_path = False
except Exception as e:
    print("Warn: could not disable Mamba fast path on CPU:", e)

# create a GPU copy if available
use_cuda = torch.cuda.is_available()
if use_cuda:
    gpu_device = torch.device('cuda')
    # new model instance for GPU to avoid .to() weight movements repeatedly
    model_gpu = create_model(margs).to(gpu_device)
    model_gpu.load_state_dict(state_dict)
    model_gpu.eval()
else:
    model_gpu = None

print("Models ready.")


Vision Mamba Config: {'img_size': (512, 512), 'patch_size': 16, 'stride': 16, 'embed_dim': 768, 'depth': 24, 'channels': 1, 'num_classes': 8, 'drop_rate': 0.0, 'drop_path_rate': 0.1, 'norm_epsilon': 1e-05, 'rms_norm': False, 'residual_in_fp32': False, 'fused_add_norm': False, 'if_rope': False, 'if_rope_residual': False, 'bimamba_type': 'v2', 'if_cls_token': True, 'if_divide_out': True, 'use_double_cls_token': False, 'use_middle_cls_token': False, 'if_bidirectional': True, 'final_pool_type': 'none', 'if_abs_pos_embed': True, 'if_bimamba': False}
Loading pretrained model from: /scratch/merileo/exp/pretrain/amba-base-f16-t16-b16-lr1e-4-m300-custom-tr0.8-full_dataset_hydrophones_FINAL/models/pretrain-joint_best_checkpoint.pth
Loaded state dict keys: dict_keys(['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'best_metrics', 'val_acc', 'args'])
Now loading SSL pretrained model from /scratch/merileo/exp/pretrain/amba-base-f16-t16-b16-lr1e-4-m300-cus

In [8]:
# Timing helpers

def time_inference(model, data_loader, device, use_amp=False, desc=None):
    model.eval()
    # warmup (GPU)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    warmup_batches = 2
    with torch.no_grad():
        it = iter(data_loader)
        for _ in range(min(warmup_batches, len(data_loader))):
            try:
                batch = next(it)
            except StopIteration:
                break
            if isinstance(batch, (list, tuple)):
                inputs = batch[0]
            else:
                inputs = batch
            inputs = inputs.to(device, non_blocking=True)
            if device.type == 'cuda' and use_amp:
                with torch.cuda.amp.autocast():
                    _ = model(inputs, task=getattr(margs, 'task', 'ft_cls'))
            else:
                _ = model(inputs, task=getattr(margs, 'task', 'ft_cls'))
        if device.type == 'cuda':
            torch.cuda.synchronize()

    # timed run
    batch_times = []
    total = 0.0
    torch.cuda.synchronize() if device.type == 'cuda' else None
    t0 = time.perf_counter()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc or 'infer', unit='batch', leave=False):
            if isinstance(batch, (list, tuple)):
                inputs = batch[0]
            else:
                inputs = batch
            inputs = inputs.to(device, non_blocking=True)
            bt0 = time.perf_counter()
            if device.type == 'cuda' and use_amp:
                with torch.cuda.amp.autocast():
                    _ = model(inputs, task=getattr(margs, 'task', 'ft_cls'))
            else:
                _ = model(inputs, task=getattr(margs, 'task', 'ft_cls'))
            torch.cuda.synchronize() if device.type == 'cuda' else None
            bt1 = time.perf_counter()
            batch_times.append(bt1 - bt0)
    total = time.perf_counter() - t0
    torch.cuda.synchronize() if device.type == 'cuda' else None
    num_items = len(getattr(data_loader.dataset, 'paths', [])) or SUBSET_SIZE
    throughput = num_items / total if total > 0 else float('inf')
    return {
        'device': str(device),
        'amp': bool(use_amp),
        'total_s': total,
        'throughput_fps': throughput,
        'mean_batch_s': float(np.mean(batch_times)) if batch_times else float('nan'),
        'p95_batch_s': float(np.percentile(batch_times, 95)) if batch_times else float('nan'),
        'batches': len(batch_times),
        'num_items': num_items,
    }



In [ ]:
# Run CPU benchmark
cpu_result = time_inference(model_cpu, cpu_loader, device=cpu_device, use_amp=False, desc='CPU')
cpu_result


RuntimeError: Expected x.is_cuda() to be true, but got false.  (Could this error message be improved?  If so, please report an enhancement request to PyTorch.)

: 

In [ ]:
# Run GPU benchmark (if available)
if use_cuda and model_gpu is not None:
    gpu_result_fp32 = time_inference(model_gpu, gpu_loader, device=gpu_device, use_amp=False, desc='GPU fp32')
    try:
        gpu_result_amp = time_inference(model_gpu, gpu_loader, device=gpu_device, use_amp=True, desc='GPU AMP')
    except Exception as e:
        print("AMP run failed:", e)
        gpu_result_amp = None
else:
    gpu_result_fp32, gpu_result_amp = None, None

(gpu_result_fp32, gpu_result_amp)


In [ ]:
# Aggregate results and display concise table
rows = []
rows.append({**cpu_result, 'label': 'CPU'})
if gpu_result_fp32 is not None:
    rows.append({**gpu_result_fp32, 'label': 'GPU fp32'})
if gpu_result_amp is not None:
    rows.append({**gpu_result_amp, 'label': 'GPU AMP'})

summary_df = pd.DataFrame(rows)[[
    'label', 'num_items', 'batches', 'total_s', 'throughput_fps', 'mean_batch_s', 'p95_batch_s', 'amp', 'device'
]].sort_values('total_s')

summary_path = OUTPUT_DIR / 'cpu_gpu_benchmark_summary.csv'
summary_df.to_csv(summary_path, index=False)
summary_df


In [ ]:
# Quick bar chart (optional)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(summary_df['label'], summary_df['throughput_fps'], color=['tab:gray','tab:blue','tab:green'][:len(summary_df)])
ax.set_ylabel('files / second')
ax.set_title('CPU vs GPU inference throughput')
for i, v in enumerate(summary_df['throughput_fps']):
    ax.text(i, v + max(0.02, 0.01*v), f"{v:.2f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Save raw timing dicts
np.save(OUTPUT_DIR / 'cpu_result.npy', cpu_result)
if gpu_result_fp32 is not None:
    np.save(OUTPUT_DIR / 'gpu_result_fp32.npy', gpu_result_fp32)
if gpu_result_amp is not None:
    np.save(OUTPUT_DIR / 'gpu_result_amp.npy', gpu_result_amp)

print('Saved summary CSV and raw results to', OUTPUT_DIR)
